# Run on Colab

1. Open this notebook in Colab: https://colab.research.google.com/github/JoshBong/latent-dispatch/blob/main/colab.ipynb (or upload the file). Pick an **L4** GPU under *Runtime → Change runtime type*, set the first code cell, then *Run all*.
2. The code runs on the Colab machine in `/content/latent-dispatch`. It comes from the `BUNDLE_B64` cell (one commit, packed into this notebook), not from GitHub.
3. Results stay on that machine, not on Drive. When the run ends, or any time after one notebook has finished, download `/content/results-<model>-<dtype>.tar.gz` from the Files panel. Then disconnect and delete the runtime: an idle runtime keeps using compute units.

**Keep the tab open.** The Colab machine's disk is wiped when its runtime ends, and Colab Pro doesn't keep running after the browser disconnects (background execution is a Pro+ feature). Keep the laptop awake. Every question is saved as soon as it finishes, so *Run all* again on the same runtime picks up where it stopped. Set `GATE_CUT = 101` before doing that (JOURNAL.md, known issues).

Stages: machine check → code → install → benchmarks → tests (including this model's real-weights check) → smoke run of every notebook + audit → full run → pack. Everything up to the tests stops the notebook on failure. The smoke gate leaves out any arm whose output is broken. The run stops there if b1 is broken or if a notebook fails during its smoke. In the full run a failing notebook doesn't stop the next one, and results are packed after every notebook.

In [ ]:
# What to run. One model per run: check its results before moving to a bigger model.
MODEL = "Qwen/Qwen3-4B"
DTYPE = "bfloat16"
SEEDS = [0]
N_PER_CELL = 100                                  # quick preliminary; None = each notebook's full design
NOTEBOOKS = ["part0_ruler", "h1_encoding", "h2_thinking"]
SKIP = {"h2_thinking": ["t1_1024"]}               # the expensive T1 control, left out of the quick pass
RUN_TESTS = True
GATE_BOXED, GATE_CUT = 50.0, 20.0                 # smoke gate, for broken output only: an arm is left out if fewer
                                                  # than half its answers are boxed or more than a fifth hit the cap
UNANSWERABLE = ("b0", "b1_only_")                 # built so the answer can't be found: no boxed check there

ROOT = "/content/latent-dispatch"

In [ ]:
import os
import subprocess
import sys

import torch


def sh(cmd):
    """Run a shell command and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)


print(sys.version)
assert torch.cuda.is_available(), "no GPU: connect to a Colab GPU server (L4)"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")
if DTYPE == "bfloat16" and not torch.cuda.is_bf16_supported():
    raise SystemExit("this GPU has no bf16 (a T4?): pick an L4 or A100")

In [ ]:
# The code this notebook runs (commit 621c3fa), packed into this cell so the Colab machine gets it with no upload,
# no token and no GitHub access. It is not updated automatically: repack it after changing code (README, Running).
BUNDLE_B64 = "..."

In [ ]:
# Unpack the code from the cell above and clone it to /content/latent-dispatch. If that folder already exists (a rerun
# on the same machine) it is reused, so saved results are kept.
import base64

BUNDLE, BUNDLE_BRANCH = "/content/latent-dispatch.bundle", "colab-run"
if not os.path.exists(ROOT):
    with open(BUNDLE, "wb") as f:
        f.write(base64.b64decode(BUNDLE_B64))
    sh(f"git clone -q -b {BUNDLE_BRANCH} {BUNDLE} {ROOT}")
os.chdir(ROOT)
sh("git log -1 --format='%h %s'")

In [ ]:
sh("pip install -q -e '.[dev]'")       # pins transformers 4.57.6; torch comes with Colab
sys.path.insert(0, ROOT)

In [ ]:
names = ["toy", "toy_d0", "musique", "hotpotqa"]
missing = [n for n in names if not os.path.exists(f"benchmark/{n}.jsonl")]
if missing:
    sh("python -m latent_dispatch.data " + " ".join(missing))
sh("cd benchmark && sha256sum -c SHA256SUMS")

In [ ]:
if RUN_TESTS:
    sh("python -m pytest -q")                                                   # fast suite, ~1 min
    sh(f"MODEL={MODEL} DTYPE={DTYPE} DEVICE=cuda python -m pytest -q -m slow tests/test_real_model.py")

In [ ]:
# Smoke: every notebook on 2 questions per cell, then the audit. The gate only catches broken output (a wrong prompt
# or stop token gives ~0% boxed, ~100% cut): such an arm is left out of the full run, and the run stops only if b1, the
# reference every arm is compared to, is broken. Milder problems are printed below and show in the final audit.
import logging

import papermill as pm

from latent_dispatch import analysis, results

log = logging.getLogger("papermill")
log.setLevel(logging.INFO)
log.propagate = False
if not log.handlers:
    log.addHandler(logging.StreamHandler(sys.stdout))
os.makedirs("notebooks/runs", exist_ok=True)
tag = MODEL.split("/")[-1]
dropped = set()


def run_notebook(name, smoke):
    out = f"notebooks/runs/{tag}-{name}{'-smoke' if smoke else ''}.ipynb"
    pm.execute_notebook(f"notebooks/{name}.ipynb", out, kernel_name="python3", log_output=True,
                        parameters=dict(MODEL=MODEL, DTYPE=DTYPE, SMOKE=smoke, SEEDS=SEEDS, N_PER_CELL=N_PER_CELL,
                                        SKIP=SKIP.get(name, []) + sorted(dropped)))


for name in NOTEBOOKS:
    run_notebook(name, smoke=True)
df = results.load_all(MODEL, DTYPE)
display(analysis.audit(df))

answer = df.text.str.rsplit("</think>", n=1).str[-1]          # forced thinking: only the text after it counts
df["unboxed"] = ~answer.str.contains("\\boxed{", regex=False) & ~df.arm.str.startswith(UNANSWERABLE)
df["capped"] = (df["cut"].isna() | df["cut"].eq(True)) if "cut" in df else True
for arm, g in df.groupby("arm"):
    bad = g[g.unboxed | g.capped]
    if not len(bad):
        continue
    broken = len(bad) >= 2 and (100 * g.unboxed.mean() > 100 - GATE_BOXED or 100 * g.capped.mean() > GATE_CUT)
    if broken:
        dropped.add(arm)
    print(f"\n{'BROKEN, left out of the full run' if broken else 'warning'}: {arm}: "
          f"{int(g.unboxed.sum())} unboxed, {int(g.capped.sum())} hit the cap, of {len(g)}")
    for _, r in bad.head(3).iterrows():
        text = r.text if len(r.text) <= 450 else r.text[:300] + " [...] " + r.text[-150:]
        print(f"  --- {r.id}  gold={r.answers}\n  {text!r}")
if "b1" in dropped:
    raise SystemExit("b1 is broken, so nothing can be compared: read the outputs above before running anything")

In [ ]:
# Full run, one notebook after another. Resumes: questions already saved (including the smoke's) are skipped. A
# notebook that fails doesn't stop the next one, and everything saved so far is packed after every notebook.
def pack():
    sh(f"tar czf /content/results-{tag}-{DTYPE}.tar.gz results notebooks/runs")


failed = {}
for name in NOTEBOOKS:
    try:
        run_notebook(name, smoke=False)
    except Exception as e:                 # the failing cell is in notebooks/runs/<model>-<name>.ipynb
        failed[name] = f"{type(e).__name__}: {str(e)[:500]}"
        print(f"\n!!! {name} failed, going on with the next notebook: {failed[name]}", flush=True)
    pack()
display(analysis.audit(results.load_all(MODEL, DTYPE)))

In [ ]:
print(f"copy {ROOT}/results/ (or /content/results-{tag}-{DTYPE}.tar.gz) into your local repo")
if dropped or failed:
    raise SystemExit(f"left out as broken: {sorted(dropped)}; notebooks that failed: {failed}")